# StoreOps Copilot: Qwen2.5-**3B** scaling run (Kaggle T4)

Same data, chat template and recipe as the 1.5B v2 run; only the base model changes (batch 1 x 8 instead of 2 x 4 for memory). Exports Q8_0 (same accuracy as F16 in the 1.5B sweep) and Q4_K_M to `sidpnyc/storeops-copilot-qwen2.5-3b`.

**Before running:** GPU T4 x2, Internet on, secret `HF_TOKEN`, input dataset `storeops-train` (latest version, 2,986 train / 157 val). Then Save Version -> Save & Run All (Commit). About 5 hours.


In [ ]:
!pip install -q unsloth

In [ ]:
# ---- settings -------------------------------------------------------------------------------
MAX_STEPS = -1            # 50 = sanity check; -1 = full run (NUM_EPOCHS)
NUM_EPOCHS = 2            # v2: validation loss was still falling after 1 epoch
HF_REPO = "sidpnyc/storeops-copilot-qwen2.5-3b"   # GGUF repo; LoRA adapter -> HF_REPO + "-lora"
BASE = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"   # scaling run: same data and recipe as v2
MAX_SEQ = 4096
SEED = 42

In [ ]:
import glob, json, os, random
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
try:
    os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
    REPORT_TO = "wandb"
    os.environ["WANDB_PROJECT"] = "storeops-copilot"
except Exception:
    REPORT_TO = "none"
print("logging to", REPORT_TO)

# Show everything attached as input, then find the two files (Kaggle sometimes renames uploads).
files = [f for f in glob.glob("/kaggle/input/**/*", recursive=True) if os.path.isfile(f)]
print("input files:", files or "NONE - attach the storeops-train dataset (+ Add Input)")
train_path = next(f for f in files if "train_text" in os.path.basename(f))
val_path = next(f for f in files if "val_text" in os.path.basename(f))
print(train_path, val_path)

In [ ]:
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE, max_seq_length=MAX_SEQ, dtype=None, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEED,
)

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files={"train": train_path, "validation": val_path})
print(ds)
# The chat template is already applied (locally, with Qwen's own template). Check one example.
print(ds["train"][0]["text"][-1200:])

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

args = SFTConfig(
    dataset_text_field="text",
    max_seq_length=MAX_SEQ,
    packing=False,
    per_device_train_batch_size=1,   # 3B: same effective batch (8), less memory per step
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=1,
    num_train_epochs=NUM_EPOCHS,
    max_steps=MAX_STEPS,
    learning_rate=2e-4,
    lr_scheduler_type="linear",
    warmup_steps=max(1, round(0.05 * (MAX_STEPS if MAX_STEPS > 0 else -(-len(ds["train"]) // 8) * NUM_EPOCHS))),  # 5%
    optim="adamw_8bit",
    weight_decay=0.0,
    fp16=not torch.cuda.is_bf16_supported(),   # T4: fp16
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50 if MAX_STEPS == 50 else 100,
    save_strategy="no",
    seed=SEED,
    output_dir="outputs",
    report_to=REPORT_TO,
)
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds["train"],
                     eval_dataset=ds["validation"], args=args)
# Loss only on assistant turns (tool calls + final answers). Qwen renders tool results as user
# turns, so they are masked too.
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                  response_part="<|im_start|>assistant\n")

# Masking check: decoded labels of one example should contain only assistant text.
labels = trainer.train_dataset[0]["labels"]
kept = [t for t in labels if t != -100]
print("trained tokens:", len(kept), "of", len(labels))
print(tokenizer.decode(kept)[:600])

In [ ]:
stats = trainer.train()
print(stats)

In [ ]:
# Loss curve for the README
import matplotlib.pyplot as plt
hist = trainer.state.log_history
json.dump(hist, open("log_history.json", "w"), indent=1)
tr = [(h["step"], h["loss"]) for h in hist if "loss" in h]
ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
plt.figure(figsize=(6, 3.5))
plt.plot(*zip(*tr), label="train")
if ev: plt.plot(*zip(*ev), "o-", label="validation")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("Qwen2.5-1.5B QLoRA"); plt.tight_layout()
plt.savefig("loss_curve.png", dpi=150); plt.show()

In [ ]:
# Sanity check: 5 held-out prompts (one per request kind). Tool calls must be well-formed JSON.
import re
FastLanguageModel.for_inference(model)
val = [json.loads(l) for l in open(val_path)]
picked = {}
for r in val:
    if r["source"] == "store" and r["kind"] not in picked:
        picked[r["kind"]] = r
ok = 0
for kind, r in picked.items():
    prompt = r["text"].split("<|im_start|>assistant\n")[0] + "<|im_start|>assistant\n"
    ids = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**ids, max_new_tokens=200, do_sample=False)
    gen = tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=False)
    calls = re.findall(r"<tool_call>\s*(.*?)\s*</tool_call>", gen, re.S)
    valid = all(isinstance(json.loads(c), dict) and "name" in json.loads(c) for c in calls) if calls else None
    names = [json.loads(c)["name"] for c in calls] if valid else []
    if kind in ("lookup", "planning", "action"):
        good = bool(valid)
    elif kind == "ambiguous":  # asking directly, or searching first, are both fine
        good = not calls or (bool(valid) and set(names) == {"search_items"})
    else:  # out_of_scope: no tool calls
        good = not calls
    ok += bool(good)
    print(f"--- {kind}: {'OK' if good else 'CHECK'}\n{gen[:400]}\n")
print(f"{ok}/{len(picked)} sanity prompts look right")

In [ ]:
# Export and upload (full run only): LoRA adapter + GGUF q4_k_m and f16 for llama.cpp.
if MAX_STEPS == -1:
    model.push_to_hub(HF_REPO + "-lora", token=HF_TOKEN, private=False)
    tokenizer.push_to_hub(HF_REPO + "-lora", token=HF_TOKEN, private=False)
    model.push_to_hub_gguf(HF_REPO, tokenizer, quantization_method=["q8_0", "q4_k_m"], token=HF_TOKEN)
    print("uploaded to https://huggingface.co/" + HF_REPO)
else:
    print("sanity run: nothing uploaded. Set MAX_STEPS = -1 for the full run.")